# 04b — Training-set model (GSE181063 training set only)

**Lymphoma AI Research** · step 4, part b · carries out sections 4–7 of the locked analysis plan (version 1.0)

1. Verifies the plan lock and every input file (SHA-256) and stops if anything changed.
2. Loads the training patients only and applies the locked QC rule (373 patients, 171 deaths).
3. Fits the elastic-net Cox model (`l1_ratio` 0.9) with 10-fold CV, takes λ_min and **freezes** it in `results/04/model_04.json`.
4. Estimates performance inside the training set: nested CV (5 folds × 5 repeats), a cross-validated Kaplan–Meier curve and the added value over IPI + COO (descriptive).
5. Gene stability (100 bootstrap resamples) and the training-only sensitivity analyses (a)–(e).
6. Writes `results/04/manifest_04b.json` with the SHA-256 of the frozen model. Step 5 cannot start without it.

**Leakage rules.** The test expression file and the external cohorts are hashed, never opened. Test and external clinical rows are dropped right after loading. Only the frozen model leaves this notebook for the test set.

**Runtime.** Roughly 10–45 minutes on the free Colab runtime; section 6 prints an estimate. Every finished unit is saved in `results/04/checkpoints_04b/`. If the connection drops, choose *Runtime → Run all* again: finished units are loaded, not recomputed.

TR: Kilitli planın 4–7. bölümlerini yalnızca eğitim setinde uygular. Modeli kurup dondurur, iç içe CV ile dürüst bir performans tahmini çıkarır, gen kararlılığına ve duyarlılık analizlerine bakar. Test ve dış set ifade dosyaları açılmaz, yalnızca SHA-256 özetleri alınır. Bağlantı koparsa **Çalışma zamanı → Tümünü çalıştır**'a yeniden bas; biten kısımlar Drive'dan yüklenir.

## 1. Setup
TR: Gerekirse `scikit-survival` ve `lifelines` paketlerini kurar, Drive'ı bağlar, klasörleri tanımlar ve paket sürümlerini yazdırır.

In [ ]:
# Install the two survival packages only if this runtime does not have them yet.
import importlib.metadata, importlib.util, subprocess, sys
_missing = [pkg for pkg, mod in (('scikit-survival', 'sksurv'), ('lifelines', 'lifelines'))
            if importlib.util.find_spec(mod) is None]
if _missing:
    _dist = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
    _loaded = {m: sys.modules[m].__version__ for m in _dist if m in sys.modules}
    print('Installing', ', '.join(_missing), '(about a minute)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    _changed = [m for m, v in _loaded.items() if importlib.metadata.version(_dist[m]) != v]
    if _changed:
        raise RuntimeError(f'pip changed packages that were already loaded ({_changed}). '
                           'Choose Runtime -> Restart session, then Run all again.')

import datetime, hashlib, json, os, pickle, platform, re, tempfile, time, warnings
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, matplotlib
import matplotlib.pyplot as plt
import sksurv, lifelines
from scipy.stats import chi2
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

PROJECT_DIR = os.environ.get('LYMPHOMA_PROJECT_DIR', '/content/drive/MyDrive/dlbcl-prognosis')
if PROJECT_DIR.startswith('/content/drive') and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
assert os.path.isdir(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'

CLEAN, PROC = f'{PROJECT_DIR}/data/clean', f'{PROJECT_DIR}/data/processed'
RES, FIG = f'{PROJECT_DIR}/results/04', f'{PROJECT_DIR}/figures/04'
CKPT = f'{RES}/checkpoints_04b'
os.makedirs(FIG, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

N_JOBS = max(1, os.cpu_count() or 1)
JOBLIB_TMP = os.path.join(tempfile.gettempdir(), 'joblib_04b')      # local disk of the runtime, not Drive
os.makedirs(JOBLIB_TMP, exist_ok=True)
T_START, RUNTIME = time.time(), {}

VERSIONS = {'python': sys.version.split()[0], 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
            'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit-learn': sklearn.__version__, 'scikit-survival': sksurv.__version__,
            'lifelines': lifelines.__version__, 'joblib': joblib.__version__, 'matplotlib': matplotlib.__version__}
for k, v in VERSIONS.items():
    print(f'{k:16s} {v}')
print('parallel workers:', N_JOBS)

## 2. Verify the plan lock and the input files
The lock file must be exactly the one recorded after 04a. The two plan files must match the hashes stored in it, and `plan_amendments.md` must have no entries. Every input file is hashed and compared with the lock; the test and external expression files are only hashed.

TR: Kilit dosyasını, iki plan dosyasını ve bütün girdi dosyalarını SHA-256 ile kontrol eder. Bir şey değiştiyse durur. Test ve dış set dosyaları açılmaz, yalnızca özetleri alınır.

In [ ]:
EXPECTED_LOCK_SHA256 = '8d87cf887e2db4770ae3273755d4901cd0d5a2a010fdc729159e007aa129bc7d'   # from progress_step4a.md
CODE_VERSION = '04b-1.0'

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

LOCK_PATH = f'{RES}/plan_lock_04.json'
assert os.path.exists(LOCK_PATH), 'results/04/plan_lock_04.json not found. Run 04a first.'
LOCK_SHA = sha256_file(LOCK_PATH)
assert LOCK_SHA == EXPECTED_LOCK_SHA256, (
    f'The lock file is not the one recorded after 04a ({EXPECTED_LOCK_SHA256[:12]} expected, {LOCK_SHA[:12]} found). Stop.')
with open(LOCK_PATH) as f:
    LOCK = json.load(f)
for kind in ('json', 'md'):
    path = f"{PROJECT_DIR}/{LOCK[f'plan_{kind}_file']}"
    assert sha256_file(path) == LOCK[f'plan_{kind}_sha256'], f'{path} differs from the locked version. Stop.'
with open(f"{PROJECT_DIR}/{LOCK['plan_json_file']}") as f:
    PLAN = json.load(f)
assert PLAN['version'] == '1.0'

AMEND_PATH = f"{PROJECT_DIR}/{PLAN['deviations']['file']}"
amend_text = open(AMEND_PATH).read() if os.path.exists(AMEND_PATH) else ''
AMENDMENTS = re.findall(r'^## Amendment \d+.*$', amend_text, flags=re.M)
if AMENDMENTS:
    raise RuntimeError(f'plan_amendments.md has {len(AMENDMENTS)} entries. This notebook implements plan v1.0 '
                       'without amendments; check before running it.')

OPENED_KEYS = ['clinical_all', 'split', 'expr_train', 'gene_table', 'qc_samples']   # everything else: hash only
NEVER_OPEN = set(PLAN['data_access']['step4_never_opens'])
assert NEVER_OPEN <= {os.path.basename(LOCK['input_files'][k]) for k in LOCK['input_files'] if k not in OPENED_KEYS}
INPUT_SHA = {}
print('Input files (SHA-256 compared with the lock):')
for key, rel in LOCK['input_files'].items():
    path = f'{PROJECT_DIR}/{rel}'
    assert os.path.exists(path), f'missing input file: {rel}'
    h = sha256_file(path)
    INPUT_SHA[key] = {'file': rel, 'sha256': h, 'status': 'match' if h == LOCK['input_sha256'][key] else 'CHANGED'}
    note = 'opened' if key in OPENED_KEYS else 'hashed only, never opened'
    print(f"  {INPUT_SHA[key]['status']:8s} {rel:46s} ({note})")
CHANGED = [k for k, v in INPUT_SHA.items() if v['status'] != 'match']
assert not CHANGED, f'Input files changed since the plan was locked: {CHANGED}. Stop.'
print(f"Plan v{PLAN['version']} lock verified ({LOCK_SHA[:12]}); no amendments; all {len(INPUT_SHA)} input files unchanged.")

## 3. Training data (primary analysis population)
Only the training rows of the clinical and QC tables are kept. Samples flagged `sex_mismatch_confident` are excluded (plan §2), and every count is checked against the 04a inventory. Patients are sorted by `sample_id` because the CV fold assignment depends on the row order.

TR: Yalnızca eğitim satırlarını yükler, kilitli QC kuralını uygular (7 örnek çıkar) ve bütün sayıları 04a envanteriyle karşılaştırır. İfade matrisi hasta × gen olarak hazırlanır.

In [ ]:
with open(f'{RES}/inventory_04a.json') as f:
    INV = json.load(f)

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'yes'])

def gene_id_text(v):
    s = str(v).strip()
    return s.split('.')[0] if re.fullmatch(r'\d+\.0+', s) else s

TIME, EVENT = PLAN['endpoints']['primary']['time'], PLAN['endpoints']['primary']['event']
FLAG_P, FLAG_S = PLAN['qc']['primary_exclusion_flag'], PLAN['qc']['sensitivity_exclusion_flag']
TRN = PLAN['cohorts']['training']

split = pd.read_csv(f'{CLEAN}/split_GSE181063.csv')
train_ids = set(split.loc[split['set'] == TRN['split'], 'sample_id'])
clin_all = pd.read_csv(f'{CLEAN}/clinical_clean_all.csv')
tr = clin_all[clin_all['sample_id'].isin(train_ids)].copy()
del clin_all                                        # test and external rows leave memory here
qc_all = pd.read_csv(f'{PROC}/qc_samples.csv')
qc_flags = qc_all.loc[qc_all['sample_id'].isin(train_ids), ['sample_id', FLAG_P, FLAG_S]].copy()
del qc_all
assert qc_flags['sample_id'].is_unique and set(qc_flags['sample_id']) == train_ids, 'QC rows do not match the training set'
qc_flags[FLAG_P], qc_flags[FLAG_S] = as_bool(qc_flags[FLAG_P]), as_bool(qc_flags[FLAG_S])

assert tr['sample_id'].is_unique and len(tr) == TRN['patients'] and int(tr[EVENT].sum()) == TRN['deaths'], \
    'training counts differ from the plan'
tr = tr.merge(qc_flags, on='sample_id', how='left', validate='one_to_one')
EXCLUDED = sorted(tr.loc[tr[FLAG_P], 'sample_id'])
assert EXCLUDED == sorted(INV['excluded_by_primary_qc']['sample_ids']), 'QC exclusions differ from 04a'
pop = tr.loc[~tr[FLAG_P]].sort_values('sample_id').reset_index(drop=True)

PP = INV['primary_population']
counts = {'patients': (len(pop), PP['patients']), 'deaths': (int(pop[EVENT].sum()), PP['deaths']),
          'ipi_missing': (int(pop['ipi'].isna().sum()), PP['ipi_missing']),
          'corr_outliers': (int(pop[FLAG_S].sum()), PP['corr_outliers_retained'])}
counts.update({f'coo_{k}': (int((pop['coo'] == k).sum()), v) for k, v in PP['coo'].items()})
bad = {k: v for k, v in counts.items() if v[0] != v[1]}
assert not bad, f'Primary population differs from the 04a inventory: {bad}'

T = pop[TIME].to_numpy(dtype=float)
E = pop[EVENT].to_numpy(dtype=int)
assert np.isfinite(T).all() and (T > 0).all() and set(np.unique(E)) <= {0, 1}
CORR_OUT = pop[FLAG_S].to_numpy(dtype=bool)
SAMPLE_IDS = pop['sample_id'].to_numpy()

expr = pd.read_parquet(f'{PROC}/expr_GSE181063_train.parquet')
if 'gene' in expr.columns:
    expr = expr.set_index('gene')
assert set(expr.columns) == train_ids, 'expression columns are not exactly the training samples'
assert expr.shape[0] == PLAN['features']['n_genes_input'], 'unexpected number of genes'
GENE_INDEX_DTYPE = str(expr.index.dtype)
GENE_STR = np.array([gene_id_text(g) for g in expr.index])
assert len(set(GENE_STR)) == len(GENE_STR), 'duplicate gene IDs'
assert all(re.fullmatch(r'\d+', g) for g in GENE_STR), 'the gene index does not look like Entrez IDs'
X = expr[list(SAMPLE_IDS)].T.to_numpy(dtype=np.float64)        # patients x genes, rows in pop order
del expr
assert np.isfinite(X).all(), 'missing or infinite expression values'

gt = pd.read_csv(f'{PROC}/gene_table_common.csv', dtype=str)
gt['gene'] = gt['gene'].map(gene_id_text)
gt = gt.drop_duplicates('gene').set_index('gene')
SYM_COLS = [c for c in ('symbol_GPL570', 'symbol') if c in gt.columns]

def symbol_of(g):
    for c in SYM_COLS:
        s = gt[c].get(g)
        if isinstance(s, str) and s.strip():
            return s.strip()
    return g

SYMBOLS = np.array([symbol_of(g) for g in GENE_STR])
_t_counts = np.unique(T, return_counts=True)[1]
TIES = {'tied_time_values': int((_t_counts > 1).sum()), 'patients_with_tied_time': int(_t_counts[_t_counts > 1].sum())}

print(f'Primary population: {len(pop)} patients, {E.sum()} deaths ({len(EXCLUDED)} excluded by {FLAG_P}).')
print(f"IPI missing: {counts['ipi_missing'][0]}; COO: " + ', '.join(f'{k[4:]} {v[0]}' for k, v in counts.items() if k.startswith('coo')))
print(f"Correlation outliers kept: {CORR_OUT.sum()}; tied OS times: {TIES['tied_time_values']} values shared by "
      f"{TIES['patients_with_tied_time']} patients (handled with Breslow ties).")
print(f"Expression: {X.shape[1]:,} genes (Entrez IDs, index dtype {GENE_INDEX_DTYPE}); symbols from {' > '.join(SYM_COLS) or 'none'}.")

## 4. Model-fitting functions
`fit_procedure` carries out plan §4–5 on any dataset (the "current fit"): variance filter (top 5,000 genes) and z-score estimated on that dataset; the penalty grid of scikit-survival (100 values, `alpha_min_ratio` 0.01); 10-fold CV stratified by event; grouped cross-validated partial-likelihood deviance as in glmnet; λ_min and λ_1se. Inside each CV fold the filter and the z-score are re-estimated on that fold's training rows only.

Two implementation details the plan does not spell out:
- **Deviance.** As in glmnet: per fold, the deviance of the fold model on all patients of the current fit minus its deviance on the fold's training patients, divided by the fold size; folds are averaged with fold-size weights. Breslow ties throughout.
- **Early end of a path.** scikit-survival ends a path when the deviance ratio changes by less than 0.1% over 5 penalties. With tied survival times its null deviance ignores the ties, so the ratio starts slightly below 0 and the rule can fire wrongly. Such stops are detected and the path is continued. Genuine stops keep the last coefficients for the remaining penalties, as glmnet does. Every case is counted and reported.

TR: Modeli kuran fonksiyonlar. Varyans filtresi ve z-skor her CV katmanında yalnızca o katmanın eğitim hastalarından hesaplanır; dışarıda bırakılan hastalar hiçbir adımı etkilemez. Son satırlar olabilirlik hesabını tanımıyla karşılaştıran kısa bir öz-kontrol.

In [ ]:
# Core numerical functions for step 4b. Pure functions: every input is passed explicitly,
# so they also run unchanged inside the parallel worker processes.
import time
import warnings

import numpy as np
from joblib import Parallel, delayed
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

NUMERICAL_ERRORS = (ArithmeticError, AssertionError)


# ---------------------------------------------------------------- likelihood and metrics
def breslow_loglik(eta, time_, event):
    """Breslow log partial likelihood; one value per column of eta (n x L)."""
    eta = np.asarray(eta, dtype=float)
    if eta.ndim == 1:
        eta = eta[:, None]
    time_ = np.asarray(time_, dtype=float)
    order = np.argsort(-time_, kind='mergesort')                    # descending time
    t_s, e_s, eta_s = time_[order], np.asarray(event).astype(bool)[order], eta[order]
    log_cum = np.logaddexp.accumulate(eta_s, axis=0)                # log sum exp over {time >= t}
    _, first, counts = np.unique(-t_s, return_index=True, return_counts=True)
    last_of_block = np.repeat(first + counts - 1, counts)           # tied times share one risk set
    return (eta_s[e_s] - log_cum[last_of_block][e_s]).sum(axis=0)


def breslow_saturated_loglik(time_, event):
    t_events = np.asarray(time_, dtype=float)[np.asarray(event).astype(bool)]
    _, d = np.unique(t_events, return_counts=True)
    return float(-np.sum(d * np.log(d)))


def breslow_deviance(eta, time_, event):
    """2 x (saturated - model) log partial likelihood, Breslow ties, unit weights.
    The quantity glmnet's coxnet.deviance() returns (computed there by the coxdev library)."""
    return 2.0 * (breslow_saturated_loglik(time_, event) - breslow_loglik(eta, time_, event))


def harrell_c(time_, event, score):
    """Harrell's C (higher score = higher risk). All-tied scores give 0.5."""
    return float(concordance_index_censored(np.asarray(event).astype(bool),
                                            np.asarray(time_, dtype=float),
                                            np.asarray(score, dtype=float))[0])


# ---------------------------------------------------------------- pre-processing (plan section 4)
def preprocess_fit(X, top_n, ddof=1):
    """Variance filter (top_n genes; all genes if top_n is None) and z-score statistics,
    estimated on the rows of the current fit only."""
    var = X.var(axis=0, ddof=ddof)
    if top_n is None or top_n >= X.shape[1]:
        idx = np.arange(X.shape[1])
    else:
        idx = np.sort(np.argsort(-var, kind='stable')[:top_n])
    mean = X[:, idx].mean(axis=0)
    sd = X[:, idx].std(axis=0, ddof=ddof)
    return {'idx': idx, 'mean': mean, 'sd': np.where(sd > 0, sd, 1.0), 'n_constant': int((sd <= 0).sum())}


def transform(X, prep):
    return (X[:, prep['idx']] - prep['mean']) / prep['sd']


# ---------------------------------------------------------------- penalty grid
def alpha_grid_from_max(alpha_max, n_alphas, alpha_min_ratio):
    """Same arithmetic as scikit-survival's automatic grid (repeated multiplication)."""
    factor = alpha_min_ratio ** (1.0 / (n_alphas - 1.0))
    grid = np.empty(n_alphas)
    grid[0] = alpha_max
    for i in range(1, n_alphas):
        grid[i] = grid[i - 1] * factor
    return grid


def breslow_alpha_max(Z, time_, event, l1_ratio):
    """Smallest penalty with all coefficients zero: max |score at beta = 0| / (n * l1_ratio).
    Used only if scikit-survival's automatic path fails numerically."""
    time_ = np.asarray(time_, dtype=float)
    order = np.argsort(-time_, kind='mergesort')
    t_s, e_s, Z_s = time_[order], np.asarray(event).astype(bool)[order], Z[order]
    _, first, counts = np.unique(-t_s, return_index=True, return_counts=True)
    last = np.repeat(first + counts - 1, counts)
    xbar = np.cumsum(Z_s, axis=0)[last] / (last + 1.0)[:, None]
    score = (Z_s[e_s] - xbar[e_s]).sum(axis=0)
    return float(np.max(np.abs(score)) / (Z.shape[0] * l1_ratio))


# ---------------------------------------------------------------- path fitting
def _coxnet(Z, y, l1_ratio, alphas, cfg):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        est = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alphas=alphas, n_alphas=cfg['n_alphas'],
                                     alpha_min_ratio=cfg['alpha_min_ratio'],
                                     max_iter=cfg['max_iter'], tol=cfg['tol']).fit(Z, y)
    return est, sum(issubclass(w.category, ConvergenceWarning) for w in caught)


def _fit_segment(Z, y, l1_ratio, seg, cfg):
    """One call along seg. On a numerical failure, keep the longest prefix of seg that fits
    (a prefix of a warm-started path is computed identically, so the search is valid)."""
    try:
        est, n_conv = _coxnet(Z, y, l1_ratio, seg, cfg)
        return est, n_conv, None
    except NUMERICAL_ERRORS:
        lo, hi, best = 0, len(seg), (None, 0)
        while hi - lo > 1:
            mid = (lo + hi) // 2
            try:
                best = _coxnet(Z, y, l1_ratio, seg[:mid], cfg)
                lo = mid
            except NUMERICAL_ERRORS:
                hi = mid
        return best[0], best[1], lo                    # lo = first failing position within seg


def new_path_info():
    return {'restarts_after_spurious_stop': 0, 'early_stop_at': None, 'failed_from': None,
            'convergence_warnings': 0}


def fit_path(Z, time_, event, l1_ratio, grid, cfg):
    """Coefficient path on a fixed decreasing grid. Returns coef (p x L; NaN after a failure),
    the number of valid penalties and a dict of events along the path.

    scikit-survival stops a path when the deviance ratio changes by < 0.1 % over 5 penalties.
    With tied times its null deviance ignores ties, so the deviance ratio starts slightly below 0
    and the rule can fire spuriously; such stops (deviance ratio < 0 at the stop) are restarted at
    the next penalty. Genuine stops (deviance ratio > 0): later penalties take the last fitted
    coefficients, as glmnet does for a fold path shorter than the lambda sequence."""
    L, p = len(grid), Z.shape[1]
    coef = np.zeros((p, L))
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    info = new_path_info()
    pos = 0
    while pos < L:
        seg = grid[pos:]
        est, n_conv, fail_in_seg = _fit_segment(Z, y, l1_ratio, seg, cfg)
        info['convergence_warnings'] += n_conv
        if est is None:                                # the first penalty of the segment fails
            info['failed_from'] = pos
            break
        k = est.coef_.shape[1]
        coef[:, pos:pos + k] = est.coef_
        end = pos + k
        target = len(seg) if fail_in_seg is None else fail_in_seg
        if k < target:                                 # stopped by scikit-survival's rule
            if est.deviance_ratio_[k - 1] < 0:
                info['restarts_after_spurious_stop'] += 1
                pos = end
                continue
            info['early_stop_at'] = end
            fill_to = L if fail_in_seg is None else pos + fail_in_seg
            coef[:, end:fill_to] = coef[:, [end - 1]]
            if fail_in_seg is not None:
                info['failed_from'] = pos + fail_in_seg
            break
        if fail_in_seg is not None:                    # reached the first failing penalty
            info['failed_from'] = end
            break
        pos = end
    n_valid = L if info['failed_from'] is None else info['failed_from']
    coef[:, n_valid:] = np.nan
    return coef, n_valid, info


def auto_path(Z, time_, event, l1_ratio, cfg):
    """Full-data fit: scikit-survival's automatic grid (cfg['n_alphas'] values) and the path."""
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    try:
        est, n_conv = _coxnet(Z, y, l1_ratio, None, cfg)
        alpha_max = float(est.alphas_[0])
    except NUMERICAL_ERRORS:
        est, n_conv, alpha_max = None, 0, breslow_alpha_max(Z, time_, event, l1_ratio)
    grid = alpha_grid_from_max(alpha_max, cfg['n_alphas'], cfg['alpha_min_ratio'])
    if est is not None:
        k = len(est.alphas_)
        assert np.allclose(est.alphas_, grid[:k], rtol=1e-10, atol=0), 'grid reconstruction mismatch'
        if k == len(grid):
            info = new_path_info()
            info['convergence_warnings'] = n_conv
            return grid, np.array(est.coef_, dtype=float), len(grid), info
    coef, n_valid, info = fit_path(Z, time_, event, l1_ratio, grid, cfg)
    return grid, coef, n_valid, info


# ---------------------------------------------------------------- cross-validation (plan section 5)
def run_parallel(tasks, n_jobs, temp_folder):
    if n_jobs == 1:
        return [f(*a, **kw) for f, a, kw in tasks]
    return Parallel(n_jobs=n_jobs, temp_folder=temp_folder, max_nbytes='1M')(tasks)


def cv_fold_task(X_cur, time_, event, train_idx, grid, l1_ratio, top_n, cfg):
    """One CV fold: pre-processing and path on the fold's training rows; Breslow deviance of every
    path coefficient on all rows of the current fit and on the fold's training rows."""
    X_tr = np.asarray(X_cur[train_idx])
    prep = preprocess_fit(X_tr, top_n, cfg['ddof'])
    coef, n_valid, info = fit_path(transform(X_tr, prep), time_[train_idx], event[train_idx],
                                   l1_ratio, grid, cfg)
    dev_diff = np.full(len(grid), np.nan)
    if n_valid > 0:
        eta = transform(np.asarray(X_cur), prep) @ coef[:, :n_valid]
        dev_diff[:n_valid] = (breslow_deviance(eta, time_, event)
                              - breslow_deviance(eta[train_idx], time_[train_idx], event[train_idx]))
    return {'dev_diff': dev_diff, 'n_valid': n_valid, 'info': info}


def cv_select(X_cur, time_, event, grid, n_valid_full, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder):
    """Grouped cross-validated partial-likelihood deviance, as in glmnet (cv.coxnet, cvstats,
    getOptcv): per fold (D_full - D_train) / fold size; fold-size-weighted mean and SE.
    Penalties at or after a numerical failure in any fit are dropped for all folds (plan section 5)."""
    folds = list(StratifiedKFold(n_splits=cfg['cv_folds'], shuffle=True,
                                 random_state=cv_seed).split(np.zeros(len(event)), event))
    res = run_parallel([delayed(cv_fold_task)(X_cur, time_, event, tr, grid, l1_ratio, top_n, cfg)
                        for tr, _ in folds], n_jobs, temp_folder)
    n_valid = min([n_valid_full] + [r['n_valid'] for r in res])
    if n_valid == 0:
        raise RuntimeError('every penalty failed numerically in at least one fit')
    w = np.array([len(te) for _, te in folds], dtype=float)
    cvraw = np.vstack([r['dev_diff'][:n_valid] for r in res]) / w[:, None]
    cvm = (w[:, None] * cvraw).sum(axis=0) / w.sum()
    cvsd = np.sqrt((w[:, None] * (cvraw - cvm) ** 2).sum(axis=0) / w.sum() / (len(folds) - 1))
    idx_min = int(np.flatnonzero(cvm <= cvm.min())[0])                 # largest lambda at the minimum
    idx_1se = int(np.flatnonzero(cvm <= cvm[idx_min] + cvsd[idx_min])[0])
    return {'cvm': cvm, 'cvsd': cvsd, 'idx_min': idx_min, 'idx_1se': idx_1se, 'n_valid': n_valid,
            'fold_info': [r['info'] for r in res], 'fold_sizes': w.astype(int).tolist()}


def fit_procedure(X_cur, time_, event, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder):
    """The whole procedure of plan sections 4-5 on one dataset (the 'current fit')."""
    t0 = time.time()
    X_cur = np.asarray(X_cur, dtype=float)
    time_, event = np.asarray(time_, dtype=float), np.asarray(event).astype(int)
    prep = preprocess_fit(X_cur, top_n, cfg['ddof'])
    grid, coef, n_valid_full, info_full = auto_path(transform(X_cur, prep), time_, event, l1_ratio, cfg)
    cv = cv_select(X_cur, time_, event, grid, n_valid_full, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder)
    n_genes_path = (np.abs(np.nan_to_num(coef)) > 0).sum(axis=0)
    return {'prep': prep, 'grid': grid, 'coef': coef, 'n_valid_full': n_valid_full, 'info_full': info_full,
            'n_genes_path': n_genes_path, 'n_patients': len(event), 'n_deaths': int(event.sum()),
            'l1_ratio': l1_ratio, 'top_n': top_n, 'cv_seed': cv_seed, 'seconds': time.time() - t0, **cv}


def model_at(proc, idx):
    """Selected genes (column positions in X), coefficients and z-score statistics at grid index idx."""
    b = proc['coef'][:, idx]
    sel = np.flatnonzero(b != 0)
    return {'gene_pos': proc['prep']['idx'][sel], 'coef': b[sel], 'mean': proc['prep']['mean'][sel],
            'sd': proc['prep']['sd'][sel], 'lambda': float(proc['grid'][idx]), 'idx': int(idx)}


def lp_of(model, X_rows):
    X_rows = np.asarray(X_rows, dtype=float)
    if len(model['coef']) == 0:
        return np.zeros(X_rows.shape[0])
    return ((X_rows[:, model['gene_pos']] - model['mean']) / model['sd']) @ model['coef']


def path_events(proc):
    """Counts of path events in the full-data fit and in all CV folds of one procedure."""
    infos = [proc['info_full']] + list(proc['fold_info'])
    return {'spurious_stop_restarts': int(sum(i['restarts_after_spurious_stop'] for i in infos)),
            'genuine_early_stops': int(sum(i['early_stop_at'] is not None for i in infos)),
            'numerical_failures': int(sum(i['failed_from'] is not None for i in infos)),
            'convergence_warnings': int(sum(i['convergence_warnings'] for i in infos)),
            'penalties_dropped': int(len(proc['grid']) - proc['n_valid'])}



# Self-check: Breslow log partial likelihood against its definition (direct loop over risk sets, with ties).
_rng = np.random.default_rng(0)
_t = np.round(_rng.exponential(30, 60)); _e = (_rng.random(60) < 0.6).astype(int); _eta = _rng.normal(0, 3, 60)
_ll = sum(_eta[i] - np.log(np.exp(_eta[_t >= _t[i]]).sum()) for i in np.flatnonzero(_e))
assert np.isclose(breslow_loglik(_eta, _t, _e)[0], _ll, rtol=1e-12)
print('likelihood self-check passed')

## 5. Settings, seeds and checkpoints
All settings come from the locked JSON; the seed formulas written in the plan are compared as text, so the code cannot drift from them. `cached()` stores each finished unit in `results/04/checkpoints_04b/` with a fingerprint of the code version, plan, data and package versions; a stored unit is reused only if its fingerprint matches.

TR: Ayarlar kilitli plandan okunur, tohum formülleri planla birebir karşılaştırılır. Biten her iş Drive'a kaydedilir; aynı kod, plan, veri ve paket sürümleriyle yeniden çalıştırınca yeniden hesaplanmaz.

In [ ]:
SEED = PLAN['master_seed']
CFG = {'l1_ratio': PLAN['model']['l1_ratio'], 'n_alphas': PLAN['model']['n_alphas'],
       'alpha_min_ratio': PLAN['model']['alpha_min_ratio'], 'cv_folds': PLAN['model']['cv_folds'],
       'top_n': PLAN['features']['variance_filter_top_n'], 'ddof': PLAN['features']['variance_ddof'],
       'max_iter': 100000, 'tol': 1e-7}               # scikit-survival defaults; the plan does not change them
assert PLAN['model']['ties'] == 'breslow' and PLAN['model']['lambda_rule_primary'] == 'lambda_min'
assert PLAN['model']['implementation'] == 'scikit-survival CoxnetSurvivalAnalysis'
assert PLAN['model']['cv_stratified_by'] == EVENT
NCV, SENS, S = PLAN['step4_training']['nested_cv'], PLAN['step4_training']['sensitivity'], PLAN['seeds']
assert S['final_model_cv_folds'] == SEED and S['stability_bootstrap'] == SEED + 300
assert S['nested_cv_outer_split'] == 'SEED + 100 + r (repeat r = 0..4)'
assert S['nested_cv_inner_folds'] == 'SEED + 200 + 10*r + j (outer fold j = 0..4)'

def seed_outer(r):
    return SEED + 100 + r

def seed_inner(r, j):
    return SEED + 200 + 10 * r + j

FP_BASE = {'code': CODE_VERSION, 'plan_json_sha256': LOCK['plan_json_sha256'], 'cfg': CFG,
           'inputs': {k: INPUT_SHA[k]['sha256'] for k in ('clinical_all', 'split', 'qc_samples', 'expr_train')},
           'packages': {k: VERSIONS[k] for k in ('numpy', 'scipy', 'scikit-learn', 'scikit-survival')}}

def cached(unit, params, compute):
    """Stored result if computed with the same code, plan, data, packages and parameters; otherwise compute and store."""
    path = f'{CKPT}/{unit}.pkl'
    fp = hashlib.sha256(json.dumps({**FP_BASE, 'unit': unit, 'params': params}, sort_keys=True,
                                   default=str).encode()).hexdigest()
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            if obj.get('fingerprint') == fp:
                return obj['result'], True
        except Exception:
            pass
    result = compute()
    with open(path + '.tmp', 'wb') as f:
        pickle.dump({'fingerprint': fp, 'result': result}, f, protocol=4)
    os.replace(path + '.tmp', path)
    return result, False

def procedure(Xs, Ts, Es, l1_ratio=None, top_n='plan', cv_seed=SEED):
    return fit_procedure(Xs, Ts, Es, CFG['l1_ratio'] if l1_ratio is None else l1_ratio,
                         CFG['top_n'] if top_n == 'plan' else top_n, cv_seed, CFG, N_JOBS, JOBLIB_TMP)

def jaccard(a, b):
    a, b = set(a), set(b)
    return float(len(a & b) / len(a | b)) if (a | b) else float('nan')

def fmt_min(sec):
    return f'{sec / 60:.0f} min' if sec >= 90 else f'{sec:.0f} s'

# Figure style: validated blue/orange series pair, diverging red for positive coefficients,
# neutral ink for all text and a light grid.
BLUE, ORANGE, RED, GREY = '#2a78d6', '#eb6834', '#e34948', '#b9b8b3'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e6e5e1'
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 9, 'axes.edgecolor': INK2,
                     'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2, 'axes.grid': True,
                     'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False, 'legend.labelcolor': INK2})

def savefig(fig, name):
    fig.savefig(f'{FIG}/{name}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

n_ckpt = len([f for f in os.listdir(CKPT) if f.endswith('.pkl')])
print(f'settings loaded; {n_ckpt} checkpoint files already in results/04/checkpoints_04b')

## 6. Final model on the full primary training population
The whole procedure on all 373 patients. λ_min (lowest mean CV deviance) defines the model; λ_1se is kept for the sensitivity analysis only. The figure shows the CV deviance curve with ±1 SE.

TR: Final model 373 hastanın tamamında kurulur; ceza değeri λ_min ile seçilir. Grafik, ceza değerine göre çapraz doğrulama sapmasını gösterir: en düşük nokta λ_min. Hücre sonunda kalan bölümlerin yaklaşık süresi yazılır.

In [ ]:
FINAL, _hit = cached('final_primary', {'cv_seed': SEED}, lambda: procedure(X, T, E))
RUNTIME['final_model_s'] = FINAL['seconds']
EV_FINAL = path_events(FINAL)
IDX_MIN, IDX_1SE, N_VALID = FINAL['idx_min'], FINAL['idx_1se'], FINAL['n_valid']
M_MIN, M_1SE = model_at(FINAL, IDX_MIN), model_at(FINAL, IDX_1SE)
AT_BOUNDARY = bool(IDX_MIN == N_VALID - 1)
NULL_MODEL = len(M_MIN['coef']) == 0

print('Final model', '(loaded from checkpoint)' if _hit else f"(fitted in {FINAL['seconds']:.0f} s)")
print(f"  penalty grid: {len(FINAL['grid'])} values, {N_VALID} valid; largest penalty {FINAL['grid'][0]:.4g}")
print(f"  lambda_min = {M_MIN['lambda']:.4g} (position {IDX_MIN + 1} of {N_VALID}): {len(M_MIN['coef'])} genes")
print(f"  lambda_1se = {M_1SE['lambda']:.4g} (position {IDX_1SE + 1} of {N_VALID}): {len(M_1SE['coef'])} genes (sensitivity only)")
print(f'  path events (full-data fit + 10 CV folds): {EV_FINAL}')
if AT_BOUNDARY:
    print('  NOTE: lambda_min is the smallest penalty on the grid. Reported; the grid is not extended (plan §5).')
if NULL_MODEL:
    print('  NULL RESULT: no gene is selected at lambda_min. Reported as such; settings are not changed (plan §5).')

cv_tab = pd.DataFrame({'position': np.arange(1, N_VALID + 1), 'lambda': FINAL['grid'][:N_VALID],
                       'log_lambda': np.log(FINAL['grid'][:N_VALID]), 'cv_deviance_mean': FINAL['cvm'],
                       'cv_deviance_se': FINAL['cvsd'], 'n_genes': FINAL['n_genes_path'][:N_VALID]})
cv_tab['is_lambda_min'] = cv_tab['position'] == IDX_MIN + 1
cv_tab['is_lambda_1se'] = cv_tab['position'] == IDX_1SE + 1
cv_tab.to_csv(f'{RES}/cv_deviance_04b.csv', index=False)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.errorbar(cv_tab['log_lambda'], cv_tab['cv_deviance_mean'], yerr=cv_tab['cv_deviance_se'], fmt='o', ms=3,
            color=BLUE, ecolor=GREY, elinewidth=0.8, capsize=0, zorder=3)
# Axis ranges: the region around the minimum. The deviance of heavily overfitted models (small penalties)
# rises far above it and is left off the plot; all values are in cv_deviance_04b.csv.
cvm, cvsd, lx = FINAL['cvm'], FINAL['cvsd'], np.log(FINAL['grid'][:N_VALID])
y_lo = cvm.min() - 2.5 * cvsd[IDX_MIN]
y_hi = cvm[0] + max(3 * (cvm[0] - cvm.min()), 2.5 * cvsd[0], 1e-6)
last = int(min(N_VALID - 1, np.flatnonzero(cvm <= y_hi).max() + 1))
pad = 0.03 * (lx[0] - lx[last]) if last > 0 else 0.5
ax.set_xlim(lx[last] - pad, lx[0] + pad)
ax.set_ylim(y_lo, y_hi)
ax.axvline(lx[IDX_MIN], color=INK2, lw=1, ls='-', zorder=1,
           label=f"λ_min{' = λ_1se' if IDX_MIN == IDX_1SE else ''}: {FINAL['n_genes_path'][IDX_MIN]} genes (primary)")
if IDX_1SE != IDX_MIN:
    ax.axvline(lx[IDX_1SE], color=INK2, lw=1, ls='--', zorder=1,
               label=f"λ_1se: {FINAL['n_genes_path'][IDX_1SE]} genes (sensitivity only)")
ax.legend(loc='best', fontsize=8)
ax.set_xlabel('log(λ)   (stronger penalty to the right)' + (
    '\nsmaller penalties not shown: their CV deviance is above this range (overfitting)' if last < N_VALID - 1 else ''))
ax.set_ylabel('CV partial-likelihood deviance\n(per patient, mean ± SE)')
top = ax.secondary_xaxis('top')
ticks = cv_tab.iloc[np.unique(np.linspace(0, last, 7).round().astype(int))]
top.set_xticks(ticks['log_lambda'])
top.set_xticklabels(ticks['n_genes'])
top.set_xlabel('genes in the model', color=INK2)
ax.set_title(f'10-fold CV, training set (n = {len(T)})', color=INK, loc='left')
savefig(fig, 'cv_deviance_04b.png')

_per = max(FINAL['seconds'], 5.0)
print(f'Expected time for the remaining sections: roughly {fmt_min(_per * 30)} to {fmt_min(_per * 55)}.')

## 7. Freeze the model
`results/04/model_04.json` stores what steps 5–6 need and nothing else: Entrez IDs, coefficients, training means and SDs of the selected genes, the penalty value and the risk-group cutoff (training median of the linear predictor, LP). The file is written once. If it already exists, the new result must be identical (or equal within floating-point tolerance) and the original file is kept; a different model stops the notebook.

TR: Model dondurulur. Dosya yalnızca bir kez yazılır; yeniden çalıştırmada farklı bir model çıkarsa not defteri durur ve hiçbir şeyin üzerine yazmaz. Tablo seçilen genleri katsayılarıyla gösterir (pozitif katsayı: yüksek ifade, daha yüksek risk).

In [ ]:
LP_TRAIN = lp_of(M_MIN, X)
_order = sorted(range(len(M_MIN['coef'])), key=lambda k: int(GENE_STR[M_MIN['gene_pos'][k]]))
MODEL = {
    'description': 'Frozen OS risk model from step 4b (plan v1.0). Research use only; not a diagnostic or treatment tool.',
    'plan_version': PLAN['version'], 'plan_json_sha256': LOCK['plan_json_sha256'],
    'training_population': {'cohort': TRN['gse'], 'split': TRN['split'], 'qc_rule': f'{FLAG_P} excluded',
                            'patients': int(len(T)), 'deaths': int(E.sum())},
    'estimator': {'implementation': 'scikit-survival CoxnetSurvivalAnalysis', 'scikit_survival_version': sksurv.__version__,
                  'l1_ratio': CFG['l1_ratio'], 'ties': 'breslow', 'n_alphas': CFG['n_alphas'],
                  'alpha_min_ratio': CFG['alpha_min_ratio'], 'tol': CFG['tol'], 'max_iter': CFG['max_iter']},
    'penalty': {'rule': 'lambda_min', 'lambda': M_MIN['lambda'], 'grid_position': IDX_MIN + 1,
                'valid_grid_size': int(N_VALID), 'at_grid_boundary': AT_BOUNDARY,
                'cv': f"{CFG['cv_folds']}-fold, stratified by {EVENT}, seed {SEED}; grouped partial-likelihood deviance"},
    'preprocessing': {'input': 'data/processed/expr_GSE181063_train.parquet (log2, quantile-normalized)',
                      'variance_filter_top_n': CFG['top_n'],
                      'z_score': 'z = (x - train_mean) / train_sd, ddof = 1, on the primary training population'},
    'gene_id': 'Entrez Gene ID, stored as text', 'expression_index_dtype': GENE_INDEX_DTYPE,
    'n_genes': int(len(M_MIN['coef'])),
    'genes': [{'entrez': str(GENE_STR[M_MIN['gene_pos'][k]]), 'symbol': str(SYMBOLS[M_MIN['gene_pos'][k]]),
               'coef': float(M_MIN['coef'][k]), 'train_mean': float(M_MIN['mean'][k]),
               'train_sd': float(M_MIN['sd'][k])} for k in _order],
    'risk_score': 'LP = sum over genes of coef * (x - train_mean) / train_sd',
    'risk_groups': {'rule': 'high if LP > cutoff, else low', 'cutoff': float(np.median(LP_TRAIN)),
                    'cutoff_definition': 'median LP of the primary training population',
                    'tertile_cutoffs_descriptive': [float(q) for q in np.quantile(LP_TRAIN, [1 / 3, 2 / 3])]},
    'training_lp': {'mean': float(LP_TRAIN.mean()), 'sd': float(LP_TRAIN.std(ddof=1)) if not NULL_MODEL else 0.0},
    'null_model': bool(NULL_MODEL),
}

MODEL_PATH = f'{RES}/model_04.json'
model_text = json.dumps(MODEL, indent=2, sort_keys=True) + '\n'
if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH) as f:
        old_text = f.read()
    old = json.loads(old_text)
    if old_text == model_text:
        FREEZE_STATUS = 'already frozen, identical result; file not rewritten'
    else:
        same = ([g['entrez'] for g in old['genes']] == [g['entrez'] for g in MODEL['genes']]
                and np.allclose([g['coef'] for g in old['genes']], [g['coef'] for g in MODEL['genes']], rtol=1e-6, atol=1e-9)
                and np.isclose(old['penalty']['lambda'], MODEL['penalty']['lambda'], rtol=1e-9))
        if not same:
            raise RuntimeError('A DIFFERENT frozen model already exists in results/04/model_04.json. '
                               'Nothing was overwritten. Stop and report this.')
        FREEZE_STATUS = 'already frozen, equal within floating-point tolerance; original file kept'
else:
    with open(MODEL_PATH, 'x') as f:
        f.write(model_text)
    FREEZE_STATUS = 'frozen now'
MODEL_SHA = sha256_file(MODEL_PATH)
print(f'Model {FREEZE_STATUS}: results/04/model_04.json')
print(f'SHA-256 {MODEL_SHA}')

GENES_TAB = pd.DataFrame(MODEL['genes'], columns=['entrez', 'symbol', 'coef', 'train_mean', 'train_sd'])
GENES_TAB['hr_per_sd'] = np.exp(GENES_TAB['coef'])
GENES_TAB = GENES_TAB.sort_values('coef', ascending=False).reset_index(drop=True)
GENES_TAB.to_csv(f'{RES}/model_genes_04b.csv', index=False)
pd.DataFrame({'entrez': GENE_STR[FINAL['prep']['idx']], 'symbol': SYMBOLS[FINAL['prep']['idx']]}).to_csv(
    f'{RES}/variance_filter_genes_04b.csv', index=False)

if not NULL_MODEL:
    show = GENES_TAB.loc[GENES_TAB['coef'].abs().sort_values(ascending=False).index[:40]].sort_values('coef')
    fig, ax = plt.subplots(figsize=(5.6, max(2.2, 0.2 * len(show) + 0.9)))
    ypos = np.arange(len(show))
    ax.barh(ypos, show['coef'], color=[RED if c > 0 else BLUE for c in show['coef']], height=0.7)
    ax.set_yticks(ypos)
    ax.set_yticklabels(show['symbol'])
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_xlabel('coefficient (log hazard ratio per SD of expression)\nred: higher risk · blue: lower risk')
    ax.set_title(f'Final model: {len(GENES_TAB)} genes at λ_min' + (' (40 largest |coef| shown)' if len(GENES_TAB) > 40 else ''),
                 color=INK, loc='left')
    ax.grid(axis='y', visible=False)
    savefig(fig, 'coefficients_04b.png')
    print(GENES_TAB.head(20).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## 8. Nested cross-validation (5 folds × 5 repeats)
In each outer training fold the whole procedure runs again (filter, z-score, grid, inner 10-fold CV). The outer test patients are z-scored with the outer training statistics and scored with that fold's model. Harrell's C is computed in every outer test fold, for λ_min and for λ_1se. This estimate is descriptive: the primary model goes to step 5 whatever it shows (plan §7).

TR: İç içe çapraz doğrulama, modelin hiç görmediği hastalardaki performansın eğitim seti içindeki dürüst tahminidir. 25 dış katmanın her birinde bütün yöntem baştan çalışır. Bu bölüm en uzun süren kısım; her katman bitince Drive'a kaydedilir.

In [ ]:
def nested_unit(outer_train, outer_test, inner_seed, l1_ratio, top_n):
    t0 = time.time()
    proc = procedure(X[outer_train], T[outer_train], E[outer_train], l1_ratio, top_n, inner_seed)
    out = {'n_train': int(len(outer_train)), 'n_test': int(len(outer_test)), 'deaths_test': int(E[outer_test].sum()),
           'n_valid': int(proc['n_valid']), 'at_boundary': bool(proc['idx_min'] == proc['n_valid'] - 1),
           **path_events(proc)}
    for rule in ('min', '1se'):
        m = model_at(proc, proc[f'idx_{rule}'])
        lp_te, lp_tr = lp_of(m, X[outer_test]), lp_of(m, X[outer_train])
        out[rule] = {'lambda': m['lambda'], 'position': m['idx'] + 1, 'n_genes': int(len(m['coef'])),
                     'genes': GENE_STR[m['gene_pos']].tolist(), 'c': harrell_c(T[outer_test], E[outer_test], lp_te),
                     'lp_test': lp_te, 'lp_train_median': float(np.median(lp_tr)),
                     'lp_train_mean': float(lp_tr.mean()),
                     'lp_train_sd': float(lp_tr.std(ddof=1)) if len(m['coef']) else 0.0}
    out['seconds'] = time.time() - t0
    return out

def outer_folds(r):
    return list(StratifiedKFold(n_splits=NCV['outer_folds'], shuffle=True,
                                random_state=seed_outer(r)).split(np.zeros(len(E)), E))

NESTED, rows, new_secs = {}, [], []
n_units, t_sec = NCV['outer_repeats'] * NCV['outer_folds'], time.time()
for r in range(NCV['outer_repeats']):
    for j, (otr, ote) in enumerate(outer_folds(r)):
        params = {'r': r, 'j': j, 'outer_seed': seed_outer(r), 'inner_seed': seed_inner(r, j),
                  'l1_ratio': CFG['l1_ratio'], 'top_n': CFG['top_n']}
        res, hit = cached(f'nested_r{r}_f{j}', params,
                          lambda: nested_unit(otr, ote, seed_inner(r, j), CFG['l1_ratio'], CFG['top_n']))
        res['test_idx'] = ote
        NESTED[(r, j)] = res
        if not hit:
            new_secs.append(res['seconds'])
        left = np.mean(new_secs) * (n_units - len(NESTED)) if new_secs else 0.0
        print(f"repeat {r + 1} fold {j + 1}: C(λ_min) {res['min']['c']:.3f} with {res['min']['n_genes']} genes, "
              f"C(λ_1se) {res['1se']['c']:.3f} with {res['1se']['n_genes']} genes"
              + ('  [checkpoint]' if hit else f"  [{res['seconds']:.0f} s; about {fmt_min(left)} left]"))
        rows.append({'repeat': r + 1, 'fold': j + 1, 'n_train': res['n_train'], 'n_test': res['n_test'],
                     'deaths_test': res['deaths_test'],
                     **{f'{k}_{rule}': res[rule][k] for rule in ('min', '1se') for k in ('lambda', 'n_genes', 'c')},
                     'lambda_min_at_boundary': res['at_boundary'], 'valid_penalties': res['n_valid'],
                     **{k: res[k] for k in ('spurious_stop_restarts', 'genuine_early_stops', 'numerical_failures',
                                            'convergence_warnings', 'penalties_dropped')}})
RUNTIME['nested_cv_s'] = time.time() - t_sec
NESTED_TAB = pd.DataFrame(rows)
NESTED_TAB.to_csv(f'{RES}/nested_cv_folds_04b.csv', index=False)

NESTED_SUM = {rule: {'mean_c': float(NESTED_TAB[f'c_{rule}'].mean()), 'sd_c': float(NESTED_TAB[f'c_{rule}'].std(ddof=1)),
                     'min_c': float(NESTED_TAB[f'c_{rule}'].min()), 'max_c': float(NESTED_TAB[f'c_{rule}'].max()),
                     'median_genes': float(NESTED_TAB[f'n_genes_{rule}'].median()),
                     'empty_models': int((NESTED_TAB[f'n_genes_{rule}'] == 0).sum())} for rule in ('min', '1se')}
NESTED_SUM['repeat1_mean_c_min'] = float(NESTED_TAB.loc[NESTED_TAB['repeat'] == 1, 'c_min'].mean())
NESTED_SUM['folds_lambda_min_at_boundary'] = int(NESTED_TAB['lambda_min_at_boundary'].sum())
NESTED_SUM['units'] = int(n_units)
with open(f'{RES}/nested_cv_summary_04b.json', 'w') as f:
    json.dump(NESTED_SUM, f, indent=2)

for rule, lab in (('min', 'λ_min (primary)'), ('1se', 'λ_1se')):
    s = NESTED_SUM[rule]
    print(f"{lab:16s} mean C {s['mean_c']:.3f} (SD {s['sd_c']:.3f}; range {s['min_c']:.3f}-{s['max_c']:.3f}); "
          f"median {s['median_genes']:.0f} genes; empty models {s['empty_models']}/{n_units}")

fig, ax = plt.subplots(figsize=(4.8, 3.4))
jit = np.random.default_rng(1).uniform(-0.12, 0.12, len(NESTED_TAB))       # horizontal jitter, display only
for k, (rule, col) in enumerate((('min', BLUE), ('1se', ORANGE))):
    ax.scatter(k + jit, NESTED_TAB[f'c_{rule}'], s=22, color=col, edgecolor='white', linewidth=0.8, zorder=3)
    m = NESTED_SUM[rule]['mean_c']
    ax.plot([k - 0.25, k + 0.25], [m, m], color=INK, lw=1.5, zorder=4)
    ax.annotate(f'mean {m:.3f}', (k + 0.28, m), va='center', color=INK, fontsize=8)
ax.axhline(0.5, color=INK2, lw=0.8, ls='--')
ax.annotate('no discrimination (0.5)', (1.9, 0.5), ha='right', va='bottom', color=INK2, fontsize=7)
ax.set_xticks([0, 1])
ax.set_xticklabels(['λ_min (primary)', 'λ_1se'])
ax.set_xlim(-0.5, 1.95)
ax.set_ylabel("Harrell's C, outer test fold")
ax.set_title(f"Nested CV: {NCV['outer_folds']} folds × {NCV['outer_repeats']} repeats", color=INK, loc='left')
ax.grid(axis='x', visible=False)
savefig(fig, 'nested_cv_c_04b.png')

## 9. Cross-validated Kaplan–Meier curve (repeat 1)
Each outer test fold of repeat 1 is split at the median LP of its own outer training fold (high = LP above the cutoff), and the five folds are pooled (Simon et al. 2011). Descriptive only: the groups come from five different fold models, so a standard log-rank p-value is not valid here and is not reported.

TR: Her hasta, kendisini hiç görmemiş bir modelle yüksek ya da düşük risk grubuna atanır ve iki grubun sağkalım eğrileri çizilir. Eğri yalnızca betimleyicidir; burada standart log-rank p-değeri geçerli olmadığı için verilmez.

In [ ]:
km_rows = []
for j in range(NCV['outer_folds']):
    res = NESTED[(0, j)]
    cut = res['min']['lp_train_median']
    for i, v in zip(res['test_idx'], res['min']['lp_test']):
        km_rows.append({'sample_id': SAMPLE_IDS[i], 'fold': j + 1, 'lp_out_of_fold': float(v),
                        'cutoff_outer_train_median': cut, 'group': 'high' if v > cut else 'low',
                        TIME: T[i], EVENT: int(E[i])})
CVKM_TAB = pd.DataFrame(km_rows).sort_values('sample_id').reset_index(drop=True)
CVKM_TAB.to_csv(f'{RES}/cv_km_repeat1_04b.csv', index=False)

def km_at(kmf, t=60.0):
    if t > kmf.timeline.max():
        return None, None, None                      # follow-up of the group does not reach t
    row = kmf.survival_function_.index.searchsorted(t, side='right') - 1
    ci = kmf.confidence_interval_survival_function_.iloc[row]
    return float(kmf.survival_function_.iloc[row, 0]), float(ci.iloc[0]), float(ci.iloc[1])

CVKM, kms = {}, []
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for grp, col, lab in (('low', BLUE, 'Low risk'), ('high', ORANGE, 'High risk')):
    d = CVKM_TAB[CVKM_TAB['group'] == grp]
    if len(d) == 0:
        CVKM[grp] = {'n': 0}
        continue
    kmf = KaplanMeierFitter(label=f'{lab} (n = {len(d)})').fit(d[TIME], d[EVENT])
    kmf.plot_survival_function(ax=ax, color=col, lw=2, ci_alpha=0.15)
    kms.append(kmf)
    s60, lo, hi = km_at(kmf)
    med = float(kmf.median_survival_time_)
    CVKM[grp] = {'n': int(len(d)), 'deaths': int(d[EVENT].sum()), 'os_5y': s60, 'os_5y_ci': [lo, hi],
                 'median_os_months': None if np.isinf(med) else med}
ax.set_xlabel('months since diagnosis')
ax.set_ylabel('overall survival')
ax.set_ylim(0, 1.02)
ax.set_title('Cross-validated KM, repeat 1 (out-of-fold risk groups)', color=INK, loc='left')
if kms:
    add_at_risk_counts(*kms, ax=ax, rows_to_show=['At risk'])
savefig(fig, 'cv_km_repeat1_04b.png')
for grp in ('high', 'low'):
    c = CVKM[grp]
    if c['n']:
        med = 'not reached' if c['median_os_months'] is None else f"{c['median_os_months']:.1f} months"
        s5 = 'n/a' if c['os_5y'] is None else f"{c['os_5y']:.1%} (95% CI {c['os_5y_ci'][0]:.1%}-{c['os_5y_ci'][1]:.1%})"
        print(f"{grp:4s}: n {c['n']}, deaths {c['deaths']}, 5-year OS {s5}, median OS {med}")
print('Empty fold models in repeat 1:', sum(NESTED[(0, j)]['min']['n_genes'] == 0 for j in range(NCV['outer_folds'])))

## 10. Added value over IPI + COO inside the training set (descriptive)
The out-of-fold LP of repeat 1, standardized with the LP mean and SD of its outer training fold, is added to Cox(IPI + COO) in patients with known IPI. Reported: likelihood-ratio test (1 df), adjusted HR per SD and the change in Harrell's C. COO is coded as in plan §11: GCB (reference), ABC, other (UNC, MHG). The confirmatory version of this test is H3 in step 7, on the test set.

TR: Gen skorunun IPI ve hücre kökenine ek bilgi katıp katmadığına eğitim seti içinde, betimleyici olarak bakar. Gen skoru dışarıda bırakılan katmanlardan gelir, yani hastanın kendi verisiyle kurulmuş bir modelden gelmez. Asıl test 7. adımda, test setinde yapılacak.

In [ ]:
parts = []
for j in range(NCV['outer_folds']):
    res = NESTED[(0, j)]['min']
    z = (res['lp_test'] - res['lp_train_mean']) / res['lp_train_sd'] if res['lp_train_sd'] > 0 else np.zeros(len(res['lp_test']))
    parts.append(pd.Series(z, index=NESTED[(0, j)]['test_idx']))
LP_OOF = pd.concat(parts).sort_index()
assert len(LP_OOF) == len(T) and (LP_OOF.index == np.arange(len(T))).all()

dav = pd.DataFrame({'T': T, 'E': E, 'ipi': pop['ipi'].to_numpy(dtype=float), 'coo': pop['coo'].to_numpy(),
                    'lp': LP_OOF.to_numpy()})
dav = dav[dav['ipi'].notna()].copy()
dav['coo_ABC'] = (dav['coo'] == 'ABC').astype(float)
dav['coo_other'] = (~dav['coo'].isin(['ABC', 'GCB'])).astype(float)
ADDED = {'n': int(len(dav)), 'deaths': int(dav['E'].sum()),
         'description': 'Cox(IPI + COO) vs Cox(IPI + COO + out-of-fold LP per SD), IPI complete cases, repeat 1; descriptive'}
if dav['lp'].std() == 0:
    ADDED['note'] = 'all fold models of repeat 1 are empty; the comparison is not possible'
    print(ADDED['note'])
else:
    base, full = ['ipi', 'coo_ABC', 'coo_other'], ['ipi', 'coo_ABC', 'coo_other', 'lp']
    m0 = CoxPHFitter().fit(dav[['T', 'E'] + base], 'T', 'E')
    m1 = CoxPHFitter().fit(dav[['T', 'E'] + full], 'T', 'E')
    lrt = float(2 * (m1.log_likelihood_ - m0.log_likelihood_))
    row = m1.summary.loc['lp']
    c0 = harrell_c(dav['T'], dav['E'], m0.predict_log_partial_hazard(dav[base]))
    c1 = harrell_c(dav['T'], dav['E'], m1.predict_log_partial_hazard(dav[full]))
    ADDED.update({'lrt_chi2': lrt, 'lrt_p': float(chi2.sf(lrt, 1)), 'hr_per_sd': float(row['exp(coef)']),
                  'hr_ci_95': [float(row['exp(coef) lower 95%']), float(row['exp(coef) upper 95%'])],
                  'c_ipi_coo': c0, 'c_ipi_coo_lp': c1, 'delta_c': c1 - c0})
    print(f"IPI complete cases: n = {ADDED['n']}, deaths = {ADDED['deaths']}")
    print(f"Adjusted HR per SD of the out-of-fold LP: {ADDED['hr_per_sd']:.2f} "
          f"(95% CI {ADDED['hr_ci_95'][0]:.2f}-{ADDED['hr_ci_95'][1]:.2f}); LRT chi2 {lrt:.2f}, p = {ADDED['lrt_p']:.2g}")
    print(f"Harrell's C: IPI + COO {c0:.3f} -> IPI + COO + LP {c1:.3f} (change {c1 - c0:+.3f})")
with open(f'{RES}/added_value_04b.json', 'w') as f:
    json.dump(ADDED, f, indent=2)

## 11. Gene stability (100 bootstrap resamples)
Patients are resampled with replacement. In each resample the variance filter and the z-score are re-estimated and the path is fitted down to the fixed λ_min of the final model. The selection frequency of each gene is for interpretation in step 8, not for selection (plan §7).

TR: Hastalar 100 kez yerine koyarak yeniden örneklenir ve her seferinde model aynı ceza değerinde yeniden kurulur. Bir genin kaç kez seçildiği, o genin ne kadar kararlı olduğunu gösterir. Bu sonuç gen seçmek için değil, 8. adımdaki yorum için kullanılır.

In [ ]:
def boot_task(X_all, T_all, E_all, rows, grid_to_min, l1_ratio, top_n, cfg):
    Xb = np.asarray(X_all[rows])
    prep = preprocess_fit(Xb, top_n, cfg['ddof'])
    coef, n_valid, info = fit_path(transform(Xb, prep), T_all[rows], E_all[rows], l1_ratio, grid_to_min, cfg)
    if n_valid < len(grid_to_min):
        return {'genes': None, 'n_genes': None, 'info': info}
    sel = prep['idx'][coef[:, -1] != 0]
    return {'genes': sel, 'n_genes': int(len(sel)), 'info': info}

N_BOOT = PLAN['step4_training']['stability_bootstrap']['resamples']
BOOT_ROWS = np.random.default_rng(S['stability_bootstrap']).integers(0, len(T), size=(N_BOOT, len(T)))
GRID_TO_MIN = FINAL['grid'][:IDX_MIN + 1]
BOOT, t_sec, BATCH = [], time.time(), 10
for b0 in range(0, N_BOOT, BATCH):
    ids = list(range(b0, min(N_BOOT, b0 + BATCH)))
    res, hit = cached(f'boot_{b0:03d}', {'resamples': ids, 'seed': S['stability_bootstrap'],
                                         'lambda_position': IDX_MIN + 1, 'lambda': M_MIN['lambda']},
                      lambda: run_parallel([delayed(boot_task)(X, T, E, BOOT_ROWS[b], GRID_TO_MIN, CFG['l1_ratio'],
                                                               CFG['top_n'], CFG) for b in ids], N_JOBS, JOBLIB_TMP))
    BOOT.extend(res)
    print(f"resamples {ids[0] + 1}-{ids[-1] + 1} done" + (' [checkpoint]' if hit else f' ({time.time() - t_sec:.0f} s so far)'))
RUNTIME['stability_s'] = time.time() - t_sec

ok = [b for b in BOOT if b['genes'] is not None]
freq = {}
for b in ok:
    for g in b['genes']:
        freq[g] = freq.get(g, 0) + 1
final_set = set(M_MIN['gene_pos'].tolist())
coef_of = dict(zip(M_MIN['gene_pos'].tolist(), M_MIN['coef'].tolist()))
STAB = pd.DataFrame({'entrez': [GENE_STR[g] for g in freq], 'symbol': [SYMBOLS[g] for g in freq],
                     'n_selected': list(freq.values()), 'in_final_model': [g in final_set for g in freq],
                     'final_coef': [coef_of.get(g, 0.0) for g in freq]},
                    columns=['entrez', 'symbol', 'n_selected', 'in_final_model', 'final_coef'])
STAB['selection_frequency'] = STAB['n_selected'] / max(len(ok), 1)
STAB = STAB.sort_values(['n_selected', 'in_final_model', 'entrez'], ascending=[False, False, True]).reset_index(drop=True)
STAB.to_csv(f'{RES}/stability_04b.csv', index=False)

sizes = np.array([b['n_genes'] for b in ok])
STAB_SUM = {'resamples': N_BOOT, 'successful': len(ok), 'failed_before_lambda_min': N_BOOT - len(ok),
            'model_size_median': float(np.median(sizes)) if len(sizes) else None,
            'model_size_iqr': [float(q) for q in np.percentile(sizes, [25, 75])] if len(sizes) else None,
            'final_genes': len(final_set),
            'final_genes_selected_in_50pct_or_more': int(STAB.loc[STAB['in_final_model'], 'selection_frequency'].ge(0.5).sum()),
            'path_events': {'resamples_with_spurious_stop_restarts': int(sum(b['info']['restarts_after_spurious_stop'] > 0 for b in BOOT)),
                            'genuine_early_stops': int(sum(b['info']['early_stop_at'] is not None for b in BOOT)),
                            'numerical_failures': int(sum(b['info']['failed_from'] is not None for b in BOOT)),
                            'convergence_warnings': int(sum(b['info']['convergence_warnings'] for b in BOOT))}}
print(f"Successful resamples: {len(ok)}/{N_BOOT}; median model size {STAB_SUM['model_size_median']} "
      f"(IQR {STAB_SUM['model_size_iqr']}); path events {STAB_SUM['path_events']}")
print(f"Genes of the final model selected in >= 50% of resamples: "
      f"{STAB_SUM['final_genes_selected_in_50pct_or_more']} of {len(final_set)}")

if len(STAB):
    top30 = STAB.head(30).iloc[::-1]
    fig, ax = plt.subplots(figsize=(5.6, max(2.4, 0.2 * len(top30) + 1.0)))
    ypos = np.arange(len(top30))
    ax.barh(ypos, 100 * top30['selection_frequency'], color=[BLUE if f else GREY for f in top30['in_final_model']], height=0.7)
    ax.set_yticks(ypos)
    ax.set_yticklabels(top30['symbol'])
    ax.set_xlim(0, 100)
    ax.set_xlabel('selected in % of bootstrap resamples')
    ax.set_title('Gene stability (30 most often selected)\nblue: in the final model · grey: not in it', color=INK, loc='left')
    ax.grid(axis='y', visible=False)
    savefig(fig, 'stability_top30_04b.png')
    print(STAB.head(15).to_string(index=False, float_format=lambda v: f'{v:.3f}'))

## 12. Training-only sensitivity analyses (plan §7.5)
(a) λ_1se; (b) pure LASSO (`l1_ratio` 1.0); (c) no variance filter (all 13,747 genes); (d) correlation outliers excluded; (e) five other CV seeds. For each: number of genes and overlap with the primary model (Jaccard index); for (b) and (c) also the nested-CV C on the folds of repeat 1 of the primary analysis. None of these models goes to the test set.

TR: Sonucun ayarlara ne kadar bağlı olduğunu gösteren duyarlılık analizleri. Hiçbiri test setine gitmez; yalnızca birincil modelin ne kadar sağlam olduğunu anlamak için. Jaccard indeksi iki gen listesinin örtüşmesidir (1 = aynı liste, 0 = hiç ortak gen yok).

In [ ]:
PRIMARY_GENES = set(GENE_STR[M_MIN['gene_pos']])
SENS_ROWS, t_sec = [], time.time()

def sens_row(code, desc, n_pat, model, boundary, nested_c=None, nested_sd=None, note=''):
    g = set(GENE_STR[model['gene_pos']])
    SENS_ROWS.append({'analysis': code, 'description': desc, 'patients': n_pat, 'n_genes': len(g),
                      'shared_with_primary': len(g & PRIMARY_GENES), 'jaccard_vs_primary': jaccard(g, PRIMARY_GENES),
                      'lambda': model['lambda'], 'lambda_position': model['idx'] + 1, 'at_grid_boundary': bool(boundary),
                      'nested_cv_c_mean': nested_c, 'nested_cv_c_sd': nested_sd, 'note': note})

sens_row('primary', 'lambda_min, l1_ratio 0.9, top 5000 genes', len(T), M_MIN, AT_BOUNDARY,
         NESTED_SUM['min']['mean_c'], NESTED_SUM['min']['sd_c'],
         f"nested CV 5x5; repeat 1 alone: C {NESTED_SUM['repeat1_mean_c_min']:.3f}")
sens_row('a', 'lambda_1se (same CV as primary)', len(T), M_1SE, IDX_1SE == N_VALID - 1,
         NESTED_SUM['1se']['mean_c'], NESTED_SUM['1se']['sd_c'], 'nested CV 5x5, lambda_1se of each fold')

def nested_repeat1(tag, l1_ratio, top_n):
    cs = []
    for j, (otr, ote) in enumerate(outer_folds(0)):
        params = {'j': j, 'outer_seed': seed_outer(0), 'inner_seed': seed_inner(0, j), 'l1_ratio': l1_ratio, 'top_n': top_n}
        res, hit = cached(f'sens_{tag}_nested_f{j}', params, lambda: nested_unit(otr, ote, seed_inner(0, j), l1_ratio, top_n))
        cs.append(res['min']['c'])
        print(f"  ({tag}) nested fold {j + 1}: C {res['min']['c']:.3f} with {res['min']['n_genes']} genes"
              + (' [checkpoint]' if hit else f" [{res['seconds']:.0f} s]"))
    return float(np.mean(cs)), float(np.std(cs, ddof=1))

SENS_C = []
for tag, desc, l1, top_n in (('b', 'pure LASSO (l1_ratio 1.0)', SENS['b_pure_lasso']['l1_ratio'], CFG['top_n']),
                             ('c', f"no variance filter (all {X.shape[1]:,} genes)", CFG['l1_ratio'],
                              SENS['c_no_variance_filter']['variance_filter_top_n'])):
    proc, hit = cached(f'sens_{tag}_final', {'l1_ratio': l1, 'top_n': top_n, 'cv_seed': SEED},
                       lambda: procedure(X, T, E, l1, top_n))
    m = model_at(proc, proc['idx_min'])
    print(f"({tag}) {desc}: {len(m['coef'])} genes" + (' [checkpoint]' if hit else f" [{proc['seconds']:.0f} s]"))
    c_mean, c_sd = nested_repeat1(tag, l1, top_n)
    SENS_C.append(c_mean)
    sens_row(tag, desc, len(T), m, proc['idx_min'] == proc['n_valid'] - 1, c_mean, c_sd,
             'nested CV 5x1 on the folds of primary repeat 1')

keep = ~CORR_OUT
proc_d, hit = cached('sens_d_final', {'excluded': int(CORR_OUT.sum()), 'cv_seed': SEED},
                     lambda: procedure(X[keep], T[keep], E[keep]))
m_d = model_at(proc_d, proc_d['idx_min'])
print(f"(d) correlation outliers excluded ({CORR_OUT.sum()} patients): {len(m_d['coef'])} genes" + (' [checkpoint]' if hit else ''))
sens_row('d', f'correlation outliers excluded (n = {int(keep.sum())})', int(keep.sum()), m_d,
         proc_d['idx_min'] == proc_d['n_valid'] - 1)

for k, s_alt in enumerate(S['alternative_cv_seeds']):
    cv_e, hit = cached(f'sens_e_seed{s_alt}', {'cv_seed': s_alt},
                       lambda: cv_select(X, T, E, FINAL['grid'], FINAL['n_valid_full'], CFG['l1_ratio'], CFG['top_n'],
                                         s_alt, CFG, N_JOBS, JOBLIB_TMP))
    m_e = model_at(FINAL, cv_e['idx_min'])
    print(f"(e{k + 1}) CV seed {s_alt}: lambda position {cv_e['idx_min'] + 1}, {len(m_e['coef'])} genes" + (' [checkpoint]' if hit else ''))
    sens_row(f'e{k + 1}', f'CV seed {s_alt} (same full-data path)', len(T), m_e, cv_e['idx_min'] == cv_e['n_valid'] - 1)
RUNTIME['sensitivity_s'] = time.time() - t_sec

SENS_TAB = pd.DataFrame(SENS_ROWS)
SENS_TAB.to_csv(f'{RES}/sensitivity_04b.csv', index=False)
print()
print(SENS_TAB[['analysis', 'description', 'n_genes', 'shared_with_primary', 'jaccard_vs_primary', 'nested_cv_c_mean']]
      .to_string(index=False, float_format=lambda v: f'{v:.3f}'))

## 13. Manifest and summary
`results/04/manifest_04b.json` records the plan and input hashes, the SHA-256 of the frozen model (the precondition for step 5), the hashes of every output, the implementation notes, package versions and run times. On a rerun the first manifest is kept and `manifest_04b_rerun.json` is written instead. The summary is also saved as `results/04/summary_04b.txt`.

TR: Manifesto ve özet. Özetin tamamını sohbete yapıştır ya da `results/04/summary_04b.txt` dosyasını yükle.

In [ ]:
IMPLEMENTATION_NOTES = [
    'Penalty selection: grouped cross-validated partial-likelihood deviance as in the current glmnet source '
    '(buildPredmat.coxnetlist, cv.coxnet, cvstats, getOptcv.glmnet). Per fold: deviance of the fold model on all patients '
    'of the current fit minus its deviance on the fold training patients, divided by the fold size; fold-size-weighted '
    'mean and SE. Deviance = 2 x (saturated - model) Breslow log partial likelihood. lambda_min = largest penalty at the '
    'minimum; lambda_1se = largest penalty with mean deviance <= minimum + its SE.',
    'Penalty grid of each current fit: scikit-survival automatic grid on the pre-processed data of that fit '
    '(100 values, alpha_min_ratio 0.01); its CV folds use the same grid.',
    'scikit-survival ends a path when the deviance ratio changes by < 0.1% over 5 penalties. With tied survival times its '
    'null deviance ignores ties, so the ratio starts slightly below 0 and the rule can fire wrongly; such stops (ratio < 0) '
    'are continued at the next penalty. Genuine stops (ratio > 0) keep the last fitted coefficients for the remaining '
    'penalties, as glmnet does for fold paths shorter than the lambda sequence. Counts are reported for every fit.',
    'Numerical failures: penalties at and after the first failure in any fold or in the full-data fit are dropped for all '
    'folds (plan section 5).',
    'CV fold assignment: primary population sorted by sample_id; StratifiedKFold(shuffle=True) on the OS event with the '
    'plan seeds; repeat 1 = r = 0 (outer seed SEED + 100).',
    'Nested CV: each outer training fold repeats the whole procedure (variance filter, z-score, grid, inner 10-fold CV); '
    'outer test patients are z-scored with the outer training statistics.',
    'CV-KM: out-of-fold LP of repeat 1; each fold split at the median in-sample LP of its outer training fold; '
    'high = LP > cutoff; no log-rank p-value.',
    'Added value (descriptive): out-of-fold LP of repeat 1 standardized with the LP mean and SD of its outer training fold; '
    'Cox models fitted with lifelines (Efron ties); COO coded GCB (reference), ABC, other (UNC, MHG).',
    'Stability: bootstrap rows drawn with replacement (seed 20261315, all 100 index vectors drawn at once); in each resample '
    'the variance filter and z-score are re-estimated and the path is fitted on the final grid down to lambda_min.',
    'Sensitivity (e): the full-data path of the primary model with the fold assignment of each alternative seed.',
    'Solver settings: scikit-survival defaults (tol 1e-7, max_iter 100000).',
]
RUNTIME['total_s'] = time.time() - T_START

def to_builtin(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(type(o))

EVENT_COLS = ['spurious_stop_restarts', 'genuine_early_stops', 'numerical_failures', 'convergence_warnings', 'penalties_dropped']
events_all = {k: int(NESTED_TAB[k].sum()) for k in EVENT_COLS}
ALL_C = list(NESTED_TAB['c_min']) + list(NESTED_TAB['c_1se']) + SENS_C + \
        [ADDED[k] for k in ('c_ipi_coo', 'c_ipi_coo_lp') if k in ADDED]
red_flags = ['a C-index above 0.90 (plan red flag): look for an error before interpreting anything'] if max(ALL_C) > 0.90 else []

top_genes = ', '.join(f'{r.symbol} ({r.coef:+.3f})' for r in
                      GENES_TAB.reindex(GENES_TAB['coef'].abs().sort_values(ascending=False).index).head(8).itertuples())
most_stable = ', '.join(f'{r.symbol} {r.selection_frequency:.0%}' for r in STAB.head(6).itertuples())
sens_line = '; '.join(f'{r.analysis}) {r.n_genes} genes, J={r.jaccard_vs_primary:.2f}'
                      + (f', C={r.nested_cv_c_mean:.3f}' if r.analysis in ('b', 'c') else '')
                      for r in SENS_TAB.itertuples() if r.analysis != 'primary')

def km_text(g):
    c = CVKM[g]
    if not c['n']:
        return f'{g} n=0'
    s5 = 'n/a' if c['os_5y'] is None else f"{c['os_5y']:.1%}"
    return f"{g} n={c['n']}, deaths={c['deaths']}, 5y OS {s5}"

if 'lrt_p' in ADDED:
    added_line = (f"Added value (descriptive, IPI complete cases n={ADDED['n']}): HR/SD {ADDED['hr_per_sd']:.2f} "
                  f"({ADDED['hr_ci_95'][0]:.2f}-{ADDED['hr_ci_95'][1]:.2f}), LRT p={ADDED['lrt_p']:.2g}, "
                  f"C {ADDED['c_ipi_coo']:.3f} -> {ADDED['c_ipi_coo_lp']:.3f}")
else:
    added_line = f"Added value: {ADDED.get('note')}"

lines = [
    '==== 04b SUMMARY ====',
    f"Plan v{PLAN['version']} lock verified ({LOCK_SHA[:12]}); inputs unchanged {len(INPUT_SHA)}/{len(INPUT_SHA)}; amendments 0",
    f"Training population: {len(T)} patients, {E.sum()} deaths ({len(EXCLUDED)} excluded by {FLAG_P}); "
    f"tied OS times: {TIES['tied_time_values']} values, {TIES['patients_with_tied_time']} patients",
    f"Final model (lambda_min): {len(M_MIN['coef'])} genes; lambda {M_MIN['lambda']:.4g}, position {IDX_MIN + 1}/{N_VALID}; "
    f"at grid boundary: {'yes' if AT_BOUNDARY else 'no'}" + ('; NULL RESULT' if NULL_MODEL else ''),
    f"  lambda_1se (sensitivity only): {len(M_1SE['coef'])} genes; path events of the final fit: {EV_FINAL}",
    f'  largest |coef|: {top_genes}' if not NULL_MODEL else '  no genes selected',
    f'Frozen model: {FREEZE_STATUS}; SHA-256 {MODEL_SHA}',
    f"Nested CV ({NCV['outer_folds']}x{NCV['outer_repeats']}): C lambda_min {NESTED_SUM['min']['mean_c']:.3f} "
    f"(SD {NESTED_SUM['min']['sd_c']:.3f}, range {NESTED_SUM['min']['min_c']:.3f}-{NESTED_SUM['min']['max_c']:.3f}), "
    f"median {NESTED_SUM['min']['median_genes']:.0f} genes, empty {NESTED_SUM['min']['empty_models']}/{n_units}; "
    f"C lambda_1se {NESTED_SUM['1se']['mean_c']:.3f} (SD {NESTED_SUM['1se']['sd_c']:.3f})",
    f'  nested-CV path events (all folds): {events_all}',
    'CV-KM repeat 1 (descriptive): ' + '; '.join(km_text(g) for g in ('high', 'low')),
    added_line,
    f"Stability: {STAB_SUM['successful']}/{N_BOOT} resamples; final genes selected in >=50%: "
    f"{STAB_SUM['final_genes_selected_in_50pct_or_more']}/{STAB_SUM['final_genes']}; most stable: {most_stable}",
    f'Sensitivity: {sens_line}',
    f"Red flags: {'; '.join(red_flags) if red_flags else 'none'}",
    f"Runtime this run: {fmt_min(RUNTIME['total_s'])}",
]
SUMMARY = '\n'.join(lines)
with open(f'{RES}/summary_04b.txt', 'w') as f:
    f.write(SUMMARY + '\n')

OUTPUT_FILES = ['model_04.json', 'model_genes_04b.csv', 'variance_filter_genes_04b.csv', 'cv_deviance_04b.csv',
                'nested_cv_folds_04b.csv', 'nested_cv_summary_04b.json', 'cv_km_repeat1_04b.csv', 'added_value_04b.json',
                'stability_04b.csv', 'sensitivity_04b.csv', 'summary_04b.txt']
FIGURES = ['cv_deviance_04b.png', 'coefficients_04b.png', 'nested_cv_c_04b.png', 'cv_km_repeat1_04b.png',
           'stability_top30_04b.png']
manifest = {
    'step': '04b', 'notebook': '04b_training_model.ipynb', 'code_version': CODE_VERSION,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'plan': {'version': PLAN['version'], 'plan_json_sha256': LOCK['plan_json_sha256'],
             'plan_md_sha256': LOCK['plan_md_sha256'], 'lock_file_sha256': LOCK_SHA, 'amendments': len(AMENDMENTS)},
    'frozen_model': {'file': 'results/04/model_04.json', 'sha256': MODEL_SHA, 'status_this_run': FREEZE_STATUS,
                     'n_genes': int(len(M_MIN['coef'])), 'lambda': M_MIN['lambda'], 'null_model': bool(NULL_MODEL)},
    'inputs_verified_against_lock': INPUT_SHA,
    'data_access': {'opened': {k: LOCK['input_files'][k] for k in OPENED_KEYS},
                    'opened_rows': 'training rows only for clinical_all and qc_samples (other rows dropped right after loading)',
                    'hashed_only_never_opened': {k: v for k, v in LOCK['input_files'].items() if k not in OPENED_KEYS}},
    'population': {'patients': int(len(T)), 'deaths': int(E.sum()), 'excluded_by_primary_qc': EXCLUDED,
                   'ipi_missing': int(pop['ipi'].isna().sum()), 'corr_outliers_retained': int(CORR_OUT.sum()),
                   'os_time_ties': TIES},
    'final_fit': {'lambda_min_position': IDX_MIN + 1, 'lambda_1se_position': IDX_1SE + 1, 'valid_grid_size': int(N_VALID),
                  'at_grid_boundary': AT_BOUNDARY, 'n_genes_lambda_1se': int(len(M_1SE['coef'])), 'path_events': EV_FINAL},
    'nested_cv': NESTED_SUM, 'nested_cv_path_events': events_all,
    'cv_km_repeat1': CVKM, 'added_value_descriptive': ADDED, 'stability': STAB_SUM,
    'sensitivity': SENS_TAB.to_dict(orient='records'),
    'red_flags': red_flags,
    'outputs': {**{f'results/04/{f}': sha256_file(f'{RES}/{f}') for f in OUTPUT_FILES if os.path.exists(f'{RES}/{f}')},
                **{f'figures/04/{f}': sha256_file(f'{FIG}/{f}') for f in FIGURES if os.path.exists(f'{FIG}/{f}')}},
    'implementation_notes': IMPLEMENTATION_NOTES,
    'environment': VERSIONS, 'parallel_workers': N_JOBS,
    'runtime_seconds_this_run': {k: round(v, 1) for k, v in RUNTIME.items()},
}
MANIFEST_PATH = f'{RES}/manifest_04b.json'
target = MANIFEST_PATH if not os.path.exists(MANIFEST_PATH) else f'{RES}/manifest_04b_rerun.json'
with open(target, 'w') as f:
    json.dump(manifest, f, indent=2, default=to_builtin)
print(SUMMARY)
print()
print('manifest written:', target.replace(PROJECT_DIR + '/', ''))
if PROJECT_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.flush_and_unmount()                         # make sure every file has reached Drive before you disconnect
    print('Drive changes flushed.')

## Next
1. Paste the summary above into the chat, or upload `results/04/summary_04b.txt`.
2. Close this Colab session: **Runtime → Disconnect and delete runtime** (saves free-tier time).
3. Do not open or change `results/04/model_04.json`. Step 5 checks its SHA-256 against the manifest before it opens the test set, and starts with the leakage check of plan §9.2 (training data only).

The last code cell flushes and unmounts Google Drive so that every file is saved. To run the notebook again, use *Run all* (section 1 mounts Drive again).

TR: Özeti sohbete yapıştır (ya da `summary_04b.txt` dosyasını yükle) ve Colab oturumunu kapat. `model_04.json` dosyasını açıp değiştirme; 5. adım onu manifestodaki özetle karşılaştırarak başlayacak. Son kod hücresi Drive'ı kaydedip ayırır; yeniden çalıştırmak için yine **Tümünü çalıştır** yeterli.